<a href="https://colab.research.google.com/github/yuprotsyk/bigdata-course/blob/main/notebooks/topic05_spark_ml_extend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Аналіз та обробка великих даних

Ю.С. Процик. Курс лекцій

# Тема 5. Налаштування та експлуатація моделей

### План

1. [Налаштування моделей (Model Tuning)](#1.-Налаштування-моделей-\(Model-Tuning\))

2. [Оцінка якості (Model Evaluation)](#2.-Оцінка-якості-\(Model-Evaluation\))

3. [Практичний приклад: Model Tuning + Evaluation](#3.-Практичний-приклад:-Model-Tuning-+-Evaluation)

4. [Моніторинг та промислова експлуатація моделей (Deployment)](#4.-Моніторинг-та-промислова-експлуатація-моделей-\(Deployment\))

5. [Корисні ресурси](#5.-Корисні-ресурси)

## 1. Налаштування моделей (Model Tuning)

Ефективність алгоритмів машинного навчання безпосередньо залежить від коректності конфігурації їхніх внутрішніх налаштувань. У Spark ML процес оптимізації моделі автоматизовано за допомогою інструментів **Model Selection**.

### Гіперпараметри та проблема перенавчання

**Гіперпараметри** — це зовнішні конфігураційні параметри `Estimator`, які встановлюються до початку процесу навчання і не змінюються алгоритмом автоматично. До них належать такі параметри, як глибина дерева (`maxDepth`), кількість базових моделей в ансамблі (`numTrees`), коефіцієнт регуляризації (`regParam`), максимальна кількість ітерацій (`maxIter`).

На відміну від параметрів моделі (наприклад, ваг у лінійній регресії), гіперпараметри визначають структуру самої моделі та її здатність до узагальнення (generalization).

**Перенавчання (Overfitting)** — це явище, за якого модель демонструє високу точність на тренувальних даних, але втрачає здатність до адекватного прогнозування на нових (тестових) спостереженнях.

| Стан моделі   | Метрика на Train | Метрика на Test | Характеристика                      |
|---------------|------------------|-----------------|-------------------------------------|
| **Underfitting**  | Низька           | Низька          | Модель занадто проста              |
| **Robust Model**  | Висока           | Висока          | Оптимальний баланс                 |
| **Overfitting**   | Дуже висока      | Низька          | Модель "запам'ятала" шум          |

Механізми запобігання перенавчанню в Spark ML:

- **Регуляризація:** використання параметрів `regParam` та `elasticNetParam` для обмеження складності моделі.

- **Контроль структури:** обмеження глибини дерев або мінімальної кількості спостережень у вузлах.

- **Валідація:** використання відкладених вибірок або крос-валідації для об'єктивної оцінки.

> **Зауваження:** Виконання налаштування гіперпараметрів безпосередньо на тестовій вибірці призводить до **витоку даних (data leakage)**. У такому разі тестова вибірка починає впливати на процес навчання, що робить результати оцінки нерелевантними.

### Інструментарій автоматизації пошуку

Для автоматичного підбору параметрів Spark ML використовує систему з трьох основних компонентів:

1. **Estimator:** алгоритм або `Pipeline`, параметри якого підлягають оптимізації.

2. **ParamGrid:** сітка можливих значень параметрів.

3. **Evaluator:** об'єкт, що визначає метрику якості для порівняння моделей.


####ParamGridBuilder

Об'єкт `ParamGridBuilder` дозволяє побудувати декартовий добуток (Cartesian product) значень параметрів. Кожна точка цієї сітки представляє окрему конфігурацію моделі.

```python
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 50, 100]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()
# Результат: 3 × 3 = 9 комбінацій → 9 моделей
```

*Для прискорення обчислень рекомендується використовувати параметр `parallelism` (доступний у Spark 3.0+), що дозволяє навчати кілька моделей у сітці паралельно на рівні кластера.*

#### CrossValidator (K-Fold Cross Validation)

`CrossValidator` забезпечує найбільш об'єктивну та стійку оцінку якості моделі. Процес передбачає розбиття даних на `k` рівних частин (фолдів):

- На кожній ітерації `k-1` частин використовуються для навчання, а 1 частина — для валідації.
- Метрики якості усереднюються за всіма `k` ітераціями.
- Найкраща модель (Best Model) автоматично перенавчається на всьому вхідному наборі даних.

#### TrainValidationSplit

Це менш ресурсомістка альтернатива крос-валідації. Вона здійснює розбиття даних на дві частини один раз (на основі параметра `trainRatio`).

- **Перевага:** значно вища швидкість обчислень.
- **Недолік:** вищий ризик отримати зміщену оцінку на вибірках малого обсягу.

![Gridsearch-Cross-Validation](https://raw.githubusercontent.com/yuprotsyk/bigdata-course/refs/heads/main/img/img05-gridsearch-cross-validation.png)

**Порівняльна характеристика методів вибору моделі**

| Критерій порівняння        | CrossValidator                          | TrainValidationSplit                |
|----------------------------|------------------------------------------|-------------------------------------|
| **Статистична надійність**     | Висока (усереднення за фолдами)          | Середня (одноразове розбиття)       |
| **Обчислювальна складність**   | Висока (в k разів більше ітерацій)       | Низька                              |
| **Рекомендований обсяг даних** | Малі та середні датасети                 | Великі датасети (Big Data)          |


## 2. Оцінка якості моделей (Model Evaluation)

Оцінка якості є фінальним і обов'язковим етапом життєвого циклу моделі, що дозволяє кількісно визначити її здатність до узагальнення на нових даних. У Spark ML для цього використовується абстракція `Evaluator`.

**Об'єкти Evaluators** приймають на вхід `DataFrame` із прогнозами та повертають єдиний скалярний показник (метрику) типу `Double` за допомогою уніфікованого методу:
`evaluator.evaluate(predictions)` → `Double`.

З версії **Spark 3.0**, основні Evaluators отримали підтримку **ваг вибірки (`weightCol`)**, що дозволяє проводити зважену оцінку якості моделі на нерівномірних розподілах.

### Спеціалізовані Evaluators Spark ML

#### RegressionEvaluator

Використовується для аналізу моделей регресії.

Очікує стовпці: прогнозоване значення (`prediction`) та істинна мітка (`label`).

| Метрика | Опис | Оптимальне значення |
|:---|:---|:---|
| `rmse` (default) | Root Mean Squared Error (середньоквадратична помилка) | ↓ мінімізація |
| `mse` | Mean Squared Error | ↓ мінімізація |
| `mae` | Mean Absolute Error (середня абсолютна помилка) | ↓ мінімізація |
| `r2` | Коефіцієнт детермінації (частка поясненої дисперсії) | ↑ максимізація (до 1.0) |

#### BinaryClassificationEvaluator

Призначений для задач бінарної класифікації. Важливою особливістю є те, що цей Evaluator використовує стовпець `rawPrediction` (сирі ймовірності або логіти), а не фінальний клас (`prediction`). Це дозволяє оцінювати модель незалежно від обраного порогу класифікації.

| Метрика| Опис | Переваги |
|:---|:---|:---|
| `areaUnderROC` (default) | Площа під ROC-кривою | Стійкість до незбалансованих класів |
| `areaUnderPR` | Площа під Precision-Recall кривою | Ефективність при сильному дисбалансі класів |

#### MulticlassClassificationEvaluator

Застосовується для класифікації з двома і більше класами. Підтримує широкий спектр метрик, включаючи `accuracy`, `f1` (за замовчуванням), `weightedPrecision` та `weightedRecall`.

#### ClusteringEvaluator

Використовується для оцінки результатів кластеризації. Основною метрикою є **Silhouette score**, що вимірює компактність кластерів та ступінь їх розділеності. Значення варіюються від -1 до 1, де 1 свідчить про ідеальну кластеризацію. Підтримує метрики відстані `squaredEuclidean` (за замовч.) та `cosine`.

### Нові типи Evaluators (Spark 3.0+)

Згідно з офіційним оновленням Spark 3.0, бібліотека була розширена двома новими типами Evaluators:

- **`MultilabelClassificationEvaluator`:** для задач, де одне спостереження може належати до кількох класів одночасно.

- **`RankingEvaluator`:** спеціалізований інструмент для оцінки рекомендаційних систем (Information Retrieval).

### Об'єктивність та методологія оцінювання моделей

Оцінка моделі на тих самих даних, що використовувалися для навчання, призводить до зміщених, нереалістично оптимістичних результатів через ефект перенавчання (overfitting).

1. **Розбиття:** Поділ вхідного `DataFrame` на тренувальну (`Train`) та тестову (`Test`) вибірки за допомогою `randomSplit()`.

2. **Навчання:** Виконання `pipeline.fit(train)` для побудови `PipelineModel`.

3. **Прогноз:** Застосування `model.transform(test)` для отримання передбачень на даних, які не брали участі в навчанні.

4. **Оцінка:** Використання `evaluator.evaluate()` для розрахунку метрик якості на тестових прогнозах.

**Запобігання витоку даних (Data Leakage):**

Використання **ML Pipelines** є критичним, оскільки воно гарантує, що всі етапи підготовки (наприклад, масштабування ознак через `StandardScaler`) базуватимуться виключно на статистиках тренувальної вибірки. Це унеможливлює потрапляння інформації з тестових даних у процес навчання.

**Важливе зауваження щодо часових даних (Time-Series):**

Якщо набір даних має часову залежність, використання стандартного випадкового розбиття (`randomSplit()`) є **неприпустимим**. Випадкове перемішування призведе до "погляду в майбутнє", коли модель навчатиметься на даних із майбутнього для передбачення минулого. У таких випадках необхідно застосовувати **хронологічне розбиття** (наприклад, навчання на даних минулого року, тестування — на поточних), щоб забезпечити репрезентативність оцінки.


## 3. Практичний приклад: Model Tuning + Evaluation

Розширюємо end-to-end приклад із попередньої теми. Замість фіксованих гіперпараметрів — автоматичний підбір через `CrossValidator`. Датасет: Titanic.

**Що робимо:**
1. Перевикористовуємо Feature Engineering Pipeline
2. Будуємо сітку гіперпараметрів `ParamGridBuilder`
3. Запускаємо `CrossValidator` — він сам знаходить найкращу комбінацію
4. Оцінюємо найкращу модель через `BinaryClassificationEvaluator` і `MulticlassClassificationEvaluator`
5. Порівнюємо з `TrainValidationSplit`

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, Imputer,
    VectorAssembler, StandardScaler
)

# --- SparkSession ---
spark = SparkSession.builder \
    .appName("SparkML") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# --- Датасет Titanic ---
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = spark.createDataFrame(pd.read_csv(url))

cols_to_use = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
df = df.select(cols_to_use) \
       .withColumn("Survived", col("Survived").cast(DoubleType())) \
       .withColumn("Pclass",   col("Pclass").cast(DoubleType())) \
       .withColumn("Age",      col("Age").cast(DoubleType())) \
       .withColumn("Fare",     col("Fare").cast(DoubleType())) \
       .withColumnRenamed("Survived", "label")

train, test = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()}, Test: {test.count()}")

# --- Feature Engineering ---
str_indexer = StringIndexer(
    inputCols=["Sex", "Embarked"],
    outputCols=["SexIdx", "EmbarkedIdx"],
    handleInvalid="keep"
)
ohe = OneHotEncoder(
    inputCols=["SexIdx", "EmbarkedIdx"],
    outputCols=["SexVec", "EmbarkedVec"]
)
imputer = Imputer(
    inputCols=["Age", "Fare"],
    outputCols=["AgeImp", "FareImp"],
    strategy="median"
)
assembler = VectorAssembler(
    inputCols=["Pclass", "SexVec", "AgeImp", "SibSp",
               "Parch", "FareImp", "EmbarkedVec"],
    outputCol="raw_features"
)
scaler = StandardScaler(
    inputCol="raw_features", outputCol="features",
    withStd=True, withMean=False
)

print("Feature Engineering визначено. Готово до навчання.")

Train: 734, Test: 157
Feature Engineering визначено. Готово до навчання.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=42)

# Повний Pipeline: Feature Engineering + модель
pipeline = Pipeline(stages=[str_indexer, ohe, imputer, assembler, scaler, rf])

# Сітка гіперпараметрів: 3 × 2 = 6 комбінацій → 6 × 3 folds = 18 моделей
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees,  [10, 50, 100]) \
    .addGrid(rf.maxDepth,  [3, 5]) \
    .build()

print(f"Комбінацій: {len(paramGrid)}, Моделей буде навчено: {len(paramGrid) * 3}")

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)

cv_model = cv.fit(train)
print("CrossValidator завершив пошук")

for params, metric in zip(paramGrid, cv_model.avgMetrics):
    print(f"numTrees={params[rf.numTrees]}, maxDepth={params[rf.maxDepth]} → AUC={metric:.4f}")

Комбінацій: 6, Моделей буде навчено: 18
CrossValidator завершив пошук
numTrees=10, maxDepth=3 → AUC=0.8601
numTrees=10, maxDepth=5 → AUC=0.8526
numTrees=50, maxDepth=3 → AUC=0.8556
numTrees=50, maxDepth=5 → AUC=0.8646
numTrees=100, maxDepth=3 → AUC=0.8589
numTrees=100, maxDepth=5 → AUC=0.8578


In [ ]:
# Найкраща модель — автоматично перенавчена на всьому train
best_model = cv_model.bestModel
best_rf    = best_model.stages[-1]  # RandomForestModel

print(f"Найкраще numTrees: {best_rf.getNumTrees}")
print(f"Найкращий maxDepth: {best_rf.getMaxDepth()}")
print(f"Важливість ознак: {best_rf.featureImportances}")

# Прогноз на тестових даних
predictions = cv_model.transform(test)

# --- BinaryClassificationEvaluator ---
auc    = evaluator.evaluate(predictions)
pr_auc = evaluator.evaluate(predictions,
                             {evaluator.metricName: "areaUnderPR"})
print(f"\nROC AUC:  {auc:.4f}")
print(f"PR AUC:   {pr_auc:.4f}")

# --- MulticlassClassificationEvaluator ---
multi_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)
accuracy = multi_eval.evaluate(predictions, {multi_eval.metricName: "accuracy"})
f1       = multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"})
print(f"Accuracy: {accuracy:.4f}")
print(f"F1:       {f1:.4f}")

Найкраще numTrees: 50
Найкращий maxDepth: 5
Важливість ознак: (11,[0,1,2,3,4,5,6,7,8,9],[0.12412590317352995,0.2951260777054749,0.2660997161909345,0.08383119190552149,0.04795636094897928,0.02886067859307147,0.1205620235749363,0.015651421988878665,0.008171892078547894,0.009614733840125471])

ROC AUC:  0.8616
PR AUC:   0.8533
Accuracy: 0.8025
F1:       0.7932


In [ ]:
from pyspark.ml.tuning import TrainValidationSplit

# TrainValidationSplit — один поділ замість K фолдів
# 6 комбінацій → лише 6 моделей (проти 18 у CrossValidator)
tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,   # 80% train, 20% validation
    parallelism=2,
    seed=42
)

tvs_model = tvs.fit(train)
tvs_predictions = tvs_model.transform(test)
tvs_auc = evaluator.evaluate(tvs_predictions)

print(f"CrossValidator  AUC: {auc:.4f}  (18 моделей, надійніше)")
print(f"TrainValSplit   AUC: {tvs_auc:.4f}  (6 моделей, швидше)")

# Збереження найкращої моделі з CrossValidator
cv_model.bestModel.write().overwrite().save("/tmp/titanic_best_model")
print("Найкращу модель збережено")

## 4. Моніторинг та промислова експлуатація моделей (Deployment)

Завершення процесу навчання моделі за допомогою методу `.fit()` є лише початковим етапом життєвого циклу машинного навчання (**ML Lifecycle**). Переведення моделі у промислову експлуатацію (production) вимагає надійної інфраструктури для збереження, версіонування та моніторингу результатів.

### Управління життєвим циклом за допомогою MLflow

Оскільки бібліотека Spark MLlib зосереджена на розподілених обчисленнях, вона не містить вбудованих графічних інтерфейсів для трекінгу експериментів. У сучасній практиці стандартом де-факто для цього став **MLflow** — відкрита платформа для керування повним життєвим циклом ML.

**Переваги інтеграції MLflow та Spark ML:**

| Функція | Технічний опис |
|---------|------|
| **Experiment Tracking** | Автоматична фіксація гіперпараметрів, метрик (AUC, RMSE) та артефактів кожного запуску |
| **Model Registry** | Централізоване сховище для версіонування та управління статусами моделей (Staging → Production) |
| **Відтворюваність** | Логування середовища виконання та коду, що гарантує ідентичність результатів при повторних запусках. |
| **Model UI** | Візуальний інтерфейс для порівняльного аналізу ефективності різних ітерацій моделі. |

```python
import mlflow
import mlflow.spark

with mlflow.start_run():
    # Логування гіперпараметрів
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)

    model = pipeline.fit(train)

    # Логування метрик
    auc = evaluator.evaluate(model.transform(test))
    mlflow.log_metric("auc", auc)

    # Збереження моделі у форматі MLflow
    mlflow.spark.log_model(model, "spark-model")
```

> [MLflow документація](https://mlflow.org/docs/latest/python_api/mlflow.spark.html)

### MLflow у Google Colab

MLflow — безкоштовний open-source інструмент (Apache 2.0). Встановлюється одною командою і працює прямо у Colab без додаткової інфраструктури.

Два режими використання:
- **Без UI** — зберігає метрики локально, достатньо для трекінгу
- **З UI** — повноцінний веб-інтерфейс через ngrok-тунель

In [ ]:
!pip install mlflow -q

import mlflow
import mlflow.spark

mlflow.set_tracking_uri("file:/tmp/mlruns")
mlflow.set_experiment("titanic_experiment")

with mlflow.start_run(run_name="rf_crossvalidator"):

    # cv_model вже навчений у попередній комірці
    best_model = cv_model.bestModel
    best_rf    = best_model.stages[-1]

    # Логування найкращих гіперпараметрів
    mlflow.log_param("numTrees",        best_rf.getNumTrees)
    mlflow.log_param("maxDepth",        best_rf.getMaxDepth())
    mlflow.log_param("numFolds",        cv.getNumFolds())
    mlflow.log_param("numCombinations", len(paramGrid))

    # Метрики по фолдах для кожної комбінації
    for i, metric in enumerate(cv_model.avgMetrics):
        mlflow.log_metric("avg_auc_combination", metric, step=i)

    # Оцінка найкращої моделі на тесті
    predictions = cv_model.transform(test)

    auc      = evaluator.evaluate(predictions)
    accuracy = multi_eval.evaluate(predictions,
                                   {multi_eval.metricName: "accuracy"})
    f1       = multi_eval.evaluate(predictions,
                                   {multi_eval.metricName: "f1"})

    mlflow.log_metric("test_auc",      auc)
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_f1",       f1)

    mlflow.spark.log_model(best_model, "best-spark-model")

    print(f"Найкраще numTrees: {best_rf.getNumTrees}")
    print(f"Найкращий maxDepth: {best_rf.getMaxDepth()}")
    print(f"Test AUC={auc:.4f} | Accuracy={accuracy:.4f} | F1={f1:.4f}")

# Перегляд запусків
runs = mlflow.search_runs(order_by=["metrics.test_auc DESC"])
print(runs[["run_id", "params.numTrees", "params.maxDepth",
            "metrics.test_auc", "metrics.test_accuracy",
            "metrics.test_f1"]])

Найкраще numTrees: 50
Найкращий maxDepth: 5
Test AUC=0.8616 | Accuracy=0.8025 | F1=0.7932
                             run_id params.numTrees params.maxDepth  \
0  29af3cb37a61455cbd3c7b69601a984e              50               5   
1  4986036785e4494ba7ee5bed7571605a              50               5   
2  8dc45ed36faa4d13a06c357dd309bddf              50               5   

   metrics.test_auc  metrics.test_accuracy  metrics.test_f1  
0          0.861559               0.802548         0.793187  
1          0.861559               0.802548         0.793187  
2          0.861559               0.802548         0.793187  


### Моніторинг у реальному часі
Починаючи з версії Spark 3.0, розробникам доступний Spark ML listener. Цей інструмент дозволяє на низькому рівні відстежувати статус конвеєрів (Pipelines), що є критично важливим для моніторингу працездатності складних розподілених систем у реальному часі.

### Експорт моделей за межі екосистеми Spark

Промислова експлуатація (deployment) навчених моделей часто стикається з проблемою ресурсної надмірності Spark Runtime. Стандартний об'єкт `PipelineModel` вимагає для виконання наявності JVM, активного `SparkContext` та використання структур `DataFrame`. Це створює високі затримки (latency) і є недоцільним для систем прогнозування в реальному часі

#### Стратегії експорту та розгортання

Для подолання залежності від важкої інфраструктури Spark застосовуються такі стратегії:

**1. Вбудована серіалізація (ML Persistence)**

Це базовий механізм Spark для збереження та завантаження конвеєрів у межах DataFrame-базованого API.

- **Крос-платформність:** Моделі, збережені у Scala/Java, можуть бути завантажені в Python (PySpark) для інференсу.

- **Обмеження:** Для завантаження та виконання моделі обов'язково потрібне наявне Spark-середовище.

**2. PMML (Predictive Model Markup Language)**

Відкритий стандарт на основі XML для обміну моделями між різними аналітичними платформами.

- **Критичне обмеження у Spark:** Експорт у PMML реалізований виключно для застарілого **RDD-based API** (`spark.mllib`).

- **Сумісність:** Не підтримує сучасні конвеєри `spark.ml` (Pipelines), що робить його непридатним для складних ланцюжків обробки ознак.

**3. MLeap: Автономний Runtime**

Спеціалізоване Open-source рішення, що дозволяє виконувати Spark ML Pipelines без залежності від бібліотек Spark.

- **Механізм:** Конвеєр експортується у портативний формат (Bundle), який виконується **легковаговим рушієм інференсу**.

- **Переваги:** Висока швидкість виконання завдяки підтримці серіалізації у **JSON** та **Protobuf**.

**4. MLflow**

Універсальна платформа для управління життєвим циклом машинного навчання.

- **Роль:** Виступає як обгортка, що дозволяє логувати параметри та метрики, які Spark ML самостійно не відстежує, та пакувати моделі для розгортання через Docker або REST API.

**Порівняльна характеристика методів розгортання моделей**

| Метод розгортання | Залежність від Spark Runtime            | Підтримка ML Pipelines     | Формат артефакту        |
|-------------------|-----------------------------------------|----------------------------|--------------------------|
| **ML Persistence**    | **Наявна** (потрібен SparkContext)           | Повна підтримка            | Parquet + JSON           |
| **PMML**              | **Відсутня**                            | Лише для RDD-моделей       | XML                      |
| **MLeap**             | **Відсутня** (автономний рушій)       | Повна підтримка            | JSON / Protobuf          |
| **MLflow**            | **Опційна** (залежить від моделі)          | Повна підтримка            | Власний (MLmodel)        |

## 5. Корисні ресурси

1. [Spark MLlib Tuning Guide](https://spark.apache.org/docs/latest/ml-tuning.html)

2. [MLlib Evaluation Metrics Guide](https://spark.apache.org/docs/latest/mllib-evaluation-metrics.html)

3. [Spark ML Listener for Tracking](https://spark.apache.org/docs/latest/monitoring.html)

4. [MLflow & Spark Integration](https://mlflow.org/docs/latest/ml/traditional-ml/sparkml/)

5. [Spark Third Party Projects](https://spark.apache.org/third-party-projects.html)

6. [MLeap Documentation](https://combust.github.io/mleap-docs/)
